# Demo local: modelo mensual con historia (12 meses)

Este notebook guía paso a paso el flujo completo de PANOPTO en un ambiente **local**
(Windows, sin cluster ni Airflow): alta de un modelo mensual con data dummy,
entrenamiento con el mes más antiguo, producción de los 11 meses siguientes,
construcción de las vistas del tablero, sincronización a PostgreSQL y arranque
del dashboard en Streamlit.

El orquestador `scripts/run_local_pipeline.py` replica la cadena de DAGs del
cluster:

| Paso | Subcomando | Equivalente en el cluster |
|---|---|---|
| Postgres portable | `postgres` | Postgres ya corre en el cluster |
| Crear tablas panopto_* | `tables` | `sql/ddl_hive.sql` (una sola vez) |
| Data dummy + config | `generate` | tablas fuente + `scripts/onboard_model.py` |
| Entrenamiento | `train` | `panopto_config_watcher` → `TrainingMode` |
| Producción ×11 | `produce` | `panopto_production_runner` → `run_model_date` |
| Vistas del tablero | `views` | `panopto_output_validator` → `build_dashboard_views` |
| Sync Hive → Postgres | `sync` | `panopto_dashboard_sync` → `pg_sync` |
| Dashboard | `dashboard` | `streamlit run panopto/dashboard/app.py` |

> **Prerequisitos del ambiente** (una sola vez): Python 3.11, Postgres portable
> descomprimido en `.local/pgsql` (cluster inicializado en `:55432` con el DDL de
> `sql/ddl_postgres.sql`), winutils/`hadoop.dll` en `.local/hadoop/bin` y el venv
> `.venv-local` con `pyspark==3.3.2`, `pandas<2`, `streamlit`, `psycopg2-binary`.

## 0. Convenciones: local vs cluster

Toda la configuración local es **paramétrica y separada** de la del cluster:

- `.env.local` — variables de entorno del stack local (Postgres `:55432`,
  warehouse local, sin Kerberos ni emails). La cargan los scripts vía
  `scripts/local_env.py` **antes** de importar `panopto`.
- `config/tables.local.json` — mismos nombres lógicos que `config/tables.json`
  pero sin el esquema `gcprmsbx_work` (todo vive en el metastore Derby local).
  Se selecciona con `PANOPTO_TABLES_JSON`.
- `scripts/local_stack.py` — helpers compartidos: `spark_conf()` (metastore
  Derby + `spark.sql.warehouse.dir`), `create_panopto_tables()`, `insert_rows()`.
- `scripts/run_local_pipeline.py` — orquestador de los subcomandos anteriores.

El switch completo local↔cluster se hace con variables de entorno; **el código
de `panopto/` no cambia**.

In [ ]:
# Setup del notebook: rutas y helper para ejecutar comandos del stack local.
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "panopto").is_dir():          # por si el kernel abrió en notebook/
    ROOT = ROOT.parent
os.chdir(ROOT)

VENV_PY = ROOT / ".venv-local" / "Scripts" / "python.exe"
if not VENV_PY.exists():                     # POSIX
    VENV_PY = ROOT / ".venv-local" / "bin" / "python"

sys.path.insert(0, str(ROOT / "scripts"))
import local_env
local_env.apply()                            # carga .env.local, HADOOP_HOME, PYSPARK_PYTHON

def run(*args, **kw):
    """Ejecuta un comando del venv local y muestra su salida (falla -> excepción)."""
    cmd = [str(VENV_PY), *args]
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, cwd=ROOT, env=os.environ.copy(),
                       capture_output=True, text=True, **kw)
    out = (r.stdout or "") + (r.stderr or "")
    print(out[-6000:])
    r.check_returncode()
    return out

MODEL_ID = "local_monthly_001"
print("repo:", ROOT)
print("venv:", VENV_PY, "existe:", VENV_PY.exists())

## 1. Arrancar PostgreSQL portable

El cluster local usa un Postgres portable en `.local/pgsql` (datos en
`.local/pgdata`, puerto `55432`). El subcomando `postgres` es idempotente:
si ya está corriendo no hace nada.

PostgreSQL cumple **dos funciones** en el flujo:

1. **Calendario y contactos**: `banamex_calendar_sync_d` (días hábiles que
   gobiernan `execution_monthly_day`/`execution_weekday`) y `model_contact`.
2. **Espejo del dashboard**: las 6 tablas que consume Streamlit
   (`panopto_metric_result`, `panopto_variable_summary`, `panopto_scoring_summary`,
   `panopto_data_availability`, `panopto_dashboard_semaphore`,
   `panopto_dashboard_model_summary`).

In [ ]:
run("scripts/run_local_pipeline.py", "postgres")

## 2. Crear las tablas `panopto_*` en el metastore local

Spark local usa un metastore **Hive/Derby embebido** (`spark-warehouse/` en la
raíz del repo). `local_stack.create_panopto_tables()` genera el DDL a partir de
`config/output_schemas.json` (campos con `partition: true` van a
`PARTITIONED BY`), equivalente al `sql/ddl_hive.sql` del cluster.

> Nota: `spark.sql.warehouse.dir` y `PANOPTO_HIVE_WAREHOUSE_DIR` deben apuntar
> a la **misma** ruta — si difieren, `AtomicParquetWriter` promueve las
> particiones a un directorio que el metastore no ve. Si esto ocurre, el
> subcomando `repair` (`MSCK REPAIR TABLE`) re-registra las particiones.

In [ ]:
run("scripts/run_local_pipeline.py", "tables")

## 3. Generar data dummy + dar de alta el modelo (onboarding)

`scripts/generate_dummy_data.py` crea las **5 tablas fuente** de un modelo
mensual y siembra su configuración (equivalente local a `onboard_model.py`):

| Tabla fuente | Rol | Contenido |
|---|---|---|
| `raw_<model>` | `raw` | Variables crudas: `raw_income`, `raw_age`, `raw_balance`, `raw_txn_count` (numéricas) + `raw_region`, `raw_segment` (categóricas) |
| `input_<model>` | `input` | Features del modelo: `in_income_log`, `in_balance_ratio`, `in_txn_freq`, `in_region_woe` |
| `transformed_<model>` | `transformed` | `tr_risk_score_lin`, `tr_score_decile` |
| `score_<model>` | `score` | Probabilidad del modelo (derivada del latente de riesgo) |
| `target_<model>` | `target` | Binaria por cohorte con **lag de 2 meses** |

Puntos clave del generador:

- **Target con desfase**: la partición del mes `m` contiene el desempeño de la
  cohorte scoreada en `m` (observado en `m + target_lag`). Se materializa desde
  `start - target_lag` para que el run del mes `t` encuentre la cohorte `t - lag`.
- **Correlación score↔target**: el target se muestrea `Bernoulli(score)` del
  mismo latente, así las métricas post-scoring (PSI del score, correlación,
  event rate) salen coherentes.
- **`--drift`**: inyecta drift gradual en los últimos meses y un pico de nulos
  intermedio, para ver AMBAR/RED en el semáforo.
- **Config**: escribe en `model_summary`, `model_table_config`,
  `variable_metadata`, `category_policy` con `process_date = primer mes` —
  eso dispara el modo entrenamiento, igual que `panopto_config_watcher`.
- **Postgres**: siembra `banamex_calendar_sync_d` (Lun–Vie hábiles) y un
  contacto en `model_contact`.

Parámetros útiles: `--clients` (clientes/mes), `--months` (historia total),
`--start-month YYYY-MM`, `--execution-day` (día hábil de corrida),
`--target-lag`, `--seed`.

In [ ]:
# 12 meses, 400 clientes/mes, target con lag 2, drift en los últimos meses.
# El primer mes queda reservado para entrenamiento; los 11 siguientes van a producción.
out = run("scripts/run_local_pipeline.py", "generate",
          "--model-id", MODEL_ID, "--clients", "400",
          "--months", "12", "--target-lag", "2", "--drift")
assert "Data dummy lista" in out

Verificación rápida: particiones generadas por tabla fuente.

In [ ]:
import local_stack
from panopto.sessions import SparkSessionBuilder

spark = SparkSessionBuilder(app_name="nb-check", extra_conf=local_stack.spark_conf()).build()
spark.sparkContext.setLogLevel("WARN")
for role in ("raw", "input", "transformed", "score", "target"):
    t = f"{role}_{MODEL_ID}"
    parts = spark.sql(f"SHOW PARTITIONS {t}").count()
    n = spark.table(t).count()
    print(f"{t:32s} particiones={parts:3d}  filas={n}")
spark.stop()

## 4. Entrenar con el mes más antiguo

`train` invoca `TrainingMode` sobre la **primera** `information_date`
(la `process_date` de la config). Esto escribe el baseline del modelo:

- `panopto_csi_psi_table` — bins de referencia para CSI/PSI,
- `panopto_category_baseline_rank` — ranking de categorías baseline,
- `panopto_metric_threshold_auto` — umbrales automáticos por variable.

En el cluster esto lo dispara `panopto_config_watcher` al detectar la nueva
`process_date`; aquí se corre explícito.

In [ ]:
run("scripts/run_local_pipeline.py", "train", "--model-id", MODEL_ID, "--months", "12")

## 5. Producción: los 11 meses siguientes

`produce` ejecuta `run_model_date` (la misma función que usa el DAG
`panopto_production_runner`) para cada `information_date` posterior a la de
entrenamiento. Cada corrida escribe:

- `panopto_metric_result` — métricas por variable (CSI, PSI, median_shift,
  population_variation, completeness, correlation_drift, event_rate…),
- `panopto_alert_aggregate` — alertas agregadas por política,
- `panopto_variable_summary` — resumen por variable (semáforo, conteos),
- `panopto_data_availability` — qué fuentes llegaron/faltaron vs `deadline_days`,
- `panopto_scoring_summary` — resumen de score del mes,
- `panopto_execution_log` — bitácora de cada corrida (SUCCESS/FAILED).

No se envían correos en local (`PANOPTO_DISABLE_EMAILS=true`).

In [ ]:
# ~1 min por mes en local. Cada línea "[produce] <modelo> @ <fecha>: SUCCESS" es una corrida.
out = run("scripts/run_local_pipeline.py", "produce",
          "--model-id", MODEL_ID, "--months", "12")
assert out.count("SUCCESS") >= 11

## 6. Vistas del tablero

`views` corre `build_dashboard_views` (misma función del DAG
`panopto_output_validator`), que materializa las dos vistas lógicas del
dashboard sobre Hive:

- `panopto_dashboard_semaphore` — semáforo variable×métrica×mes,
- `panopto_dashboard_model_summary` — resumen por modelo×mes.

In [ ]:
run("scripts/run_local_pipeline.py", "views")

## 7. Sincronizar Hive → PostgreSQL

`sync` ejecuta `pg_sync.sync_dashboard`: diff incremental por partición
`(information_date, model_id)` — compara `GROUP BY` + `COUNT(*)` entre Hive y
Postgres, reemplaza particiones nuevas o distintas y borra huérfanas. Crea las
tablas espejo si no existen.

El dashboard **solo lee de Postgres** (`dashboard/data.py` usa
`PostgresSession`, sin Spark), así que este paso es obligatorio antes de
abrir Streamlit. En el cluster lo dispara `panopto_dashboard_sync` por trigger
de los DAGs productores; en local es un paso manual del pipeline.

In [ ]:
out = run("scripts/run_local_pipeline.py", "sync", "--model-id", MODEL_ID)
assert "sincronizadas" in out

Verificación: conteos en las tablas espejo de Postgres (11 particiones por tabla).

In [ ]:
from panopto.sessions import PostgresSession

psql = PostgresSession()
with psql.connection() as conn, conn.cursor() as cur:
    for t in ("panopto_metric_result", "panopto_variable_summary",
              "panopto_scoring_summary", "panopto_data_availability",
              "panopto_dashboard_semaphore", "panopto_dashboard_model_summary"):
        cur.execute(f"SELECT COUNT(*) FROM {t} WHERE model_id = %s", (MODEL_ID,))
        print(f"{t:38s} {cur.fetchone()[0]:6d} filas")

    # Distribución del semáforo — con --drift deberían aparecer AMBAR/RED.
    cur.execute(
        "SELECT semaphore, COUNT(*) FROM panopto_dashboard_semaphore"
        " WHERE model_id = %s GROUP BY semaphore ORDER BY semaphore", (MODEL_ID,))
    print("\nsemaphore:", {r[0]: r[1] for r in cur.fetchall()})

## 8. Levantar el dashboard (Streamlit)

El dashboard corre sin Spark: `data.py` consulta Postgres con
`pd.read_sql` parametrizado. Para lanzarlo desde una terminal (proceso
de larga duración, no conviene dentro del notebook):

```bash
# Git Bash / cmd, desde la raíz del repo
./.venv-local/Scripts/python.exe -m streamlit run panopto/dashboard/app.py --server.port 8501
# o vía el orquestador (arranca postgres si falta y lanza streamlit)
./.venv-local/Scripts/python.exe scripts/run_local_pipeline.py dashboard --port 8501
```

URL: **http://localhost:8501**

La celda siguiente lanza streamlit en background solo para verificar que
responde; detén el proceso si ya lo tienes corriendo en otra terminal.

In [ ]:
import urllib.request

try:
    code = urllib.request.urlopen("http://localhost:8501", timeout=5).status
    print(f"dashboard ya responde en http://localhost:8501 (HTTP {code})")
except Exception:
    proc = subprocess.Popen(
        [str(VENV_PY), "-m", "streamlit", "run",
         str(ROOT / "panopto" / "dashboard" / "app.py"),
         "--server.port", "8501", "--server.headless", "true"],
        cwd=ROOT, env=os.environ.copy(),
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import time
    for _ in range(30):
        try:
            code = urllib.request.urlopen("http://localhost:8501", timeout=2).status
            break
        except Exception:
            time.sleep(1)
    print(f"streamlit levantado (PID {proc.pid}) -> http://localhost:8501 (HTTP {code})")
    print("detener: proc.terminate()")

## 9. Todo en un comando

Para regenerar el demo completo desde cero (data nueva con otra semilla u otro
rango), `all` encadena postgres → tables → generate → train → produce → views →
sync:

```bash
./.venv-local/Scripts/python.exe scripts/run_local_pipeline.py all \
    --model-id local_monthly_002 --clients 500 --months 12 \
    --start-month 2025-10 --target-lag 2 --drift --seed 7
```

## Troubleshooting

| Síntoma | Causa típica | Fix |
|---|---|---|
| `hadoop.dll`/`winutils` errors en Spark | falta `.local/hadoop/bin` en `PATH`/`java.library.path` | `local_env.apply()` ya lo fija; verificar que existan los binarios |
| Postgres no conecta | no está corriendo o puerto distinto | `run_local_pipeline.py postgres`; revisar `PANOPTO_POSTGRES_PORT` en `.env.local` |
| Sync no copia nada / tablas vacías en Postgres | `spark.sql.warehouse.dir` ≠ `PANOPTO_HIVE_WAREHOUSE_DIR` | alinearlos en `.env.local`/`local_stack.py` y correr `repair` + `sync` |
| Derby `metastore_db` lock | un Spark anterior quedó vivo | cerrar sesiones Spark previas; borrar `metastore_db/db.lck` |
| `AttributeError: iteritems` | pandas ≥2 con pyspark 3.3 | usar `pandas<2` en `.venv-local` |
| Re-correr produce duplica meses | particiones ya existen | `pg_sync` es idempotente por partición; en Hive `AtomicParquetWriter` sobreescribe la partición |
| Cambios de config (p.ej. `reading_mode`) no se reflejan al re-correr | `Checkpoint` cachea resultados por clave `(model_id, information_date, process_dates)` en `.local/checkpoints` | `rm -rf .local/checkpoints` y volver a `produce` |
| `completeness` siempre RED (~0.95) con `baseline` vacío | `reading_mode="each"` espera una partición por día hábil del mes, pero la fuente es snapshot mensual (1 partición); el `baseline` es `None` por diseño (compara fechas vs calendario esperado) | usar `reading_mode="first_partition"` (o `first`/`last`) para fuentes de una partición por periodo |